# Code Review Bug Drills (Blitzy Round 1)

Covers: *"Read a short function, identify the bug, walk through how you'd fix and test it."*

**How to use this notebook**
1. Read the **buggy code** cell and try to spot the bug yourself *before* running it.
2. Run the cell. It shows the failure happening.
3. Read **Bug → Breaking input → Fix → Tests** and say them out loud.
4. Run the **fix** and **test** cells. Every test should print `✅`.

Use **Run All** to check every fix in one go. The last cell prints a summary.

## The 4-step answer script (say this out loud)

- **1. Read & restate (30s):** "This function takes X and is meant to return Y. Where is it called from? How hot is the path?"
- **2. Find the bug (1–2 min):** name the **input that breaks it**, never just "this looks wrong."
- **3. Fix (1–2 min):** the smallest correct change first, then mention a structural improvement.
- **4. Test (1 min):** a regression test for the exact failing input, plus edge cases (empty, one item, boundary, concurrency, failure).
- **Bonus (30s):** "How did this ship? Which test or review gate would have caught it?" ← the FDE / Blitzy signal.

## Scan checklist (in this order)

- **Boundaries:** off-by-one, empty input, last batch dropped.
- **State:** mutable defaults, shared globals, changing a collection while looping over it.
- **Concurrency:** check-then-act, non-atomic `+=`, missing lock.
- **Failure paths:** swallowed exceptions, retrying errors that will never succeed, no backoff, leaked resources.
- **Data:** floats for money, timezone-naive datetimes, missing `None` checks.
- **Security:** SQL built with f-strings, cache key missing the tenant or user.
- **Distributed:** non-idempotent handlers, N+1 queries, unbounded retries.

In [2]:
# Shared imports and a tiny test helper used by every drill
import json, os, random, sqlite3, tempfile, threading, time, functools
from datetime import datetime, timedelta, timezone
from decimal import Decimal, ROUND_HALF_UP

RESULTS = {}

def check(drill, name, condition):
    RESULTS.setdefault(drill, []).append(bool(condition))
    print(("✅" if condition else "❌"), f"[{drill}] {name}")

def raises(exc_type, fn, *args, **kwargs):
    try:
        fn(*args, **kwargs)
    except exc_type:
        return True
    return False

---
## Drill 1: Mutable default argument

**Buggy code.** Spot the bug before running it.

In [3]:
def add_tag_buggy(tag, tags=[]):
    tags.append(tag)
    return tags

print(add_tag_buggy("a"))
print(add_tag_buggy("b"))   # expected ['b']

['a']
['a', 'b']


- **Bug:** Python creates the default list **once**, when the function is defined, so every call shares it.
- **Breaking input:** `add_tag("a")` → `['a']`, then `add_tag("b")` → `['a', 'b']`.
- **Fix:** default to `None` and create a fresh list inside; copy the caller's list so you don't mutate it.
- **Tests:** two calls without an argument are independent; a list passed in is not changed.

In [4]:
def add_tag(tag, tags=None):
    tags = [] if tags is None else list(tags)
    tags.append(tag)
    return tags

check(1, "calls are independent", add_tag("a") == ["a"] and add_tag("b") == ["b"])
caller = ["x"]
add_tag("y", caller)
check(1, "caller's list not mutated", caller == ["x"])

✅ [1] calls are independent
✅ [1] caller's list not mutated


---
## Drill 2: Off-by-one in batching

**Buggy code.**

In [ ]:
def batches_buggy(items, size):
    return [items[i:i + size] for i in range(0, len(items) - 1, size)]

print(batches_buggy([1, 2, 3], 2))   # expected [[1, 2], [3]]

[[1, 2]]


- **Bug:** `len(items) - 1` stops the range one short, so a final batch with a single item is dropped.
- **Breaking input:** `batches([1, 2, 3], 2)` → `[[1, 2]]`. The `3` is silently lost.
- **Fix:** `range(0, len(items), size)`, and reject `size <= 0` (otherwise `range` raises or loops oddly).
- **Tests:** empty list, `len == size`, `len == size + 1`, `size = 0` raises, and flattening the result gives back the input.

In [6]:
def batches(items, size):
    if size <= 0:
        raise ValueError("size must be positive")
    return [items[i:i + size] for i in range(0, len(items), size)]

check(2, "empty list", batches([], 2) == [])
check(2, "last single item kept", batches([1, 2, 3], 2) == [[1, 2], [3]])
check(2, "exact fit", batches([1, 2], 2) == [[1, 2]])
check(2, "size 0 raises", raises(ValueError, batches, [1], 0))
check(2, "flatten == input", sum(batches(list(range(7)), 3), []) == list(range(7)))

✅ [2] empty list
✅ [2] last single item kept
✅ [2] exact fit
✅ [2] size 0 raises
✅ [2] flatten == input


In [ ]:
def batches(items, size):
    if size <= 0:
        raise ValueError("size must be positive")
    return [items[i:i + size] for i in range(0, len(items), size)]

items = [1, 2, 3]
size = 2

print("indexes:", list(range(0, len(items), size)))
for i in range(0, len(items), size):
    print(f"i={i}  items[{i}:{i + size}] =", items[i:i + size])

print("result:", batches(items, size))
print("empty:", batches([], 2))
print("exact fit:", batches([1, 2], 2))
print("seven items, size 3:", batches(list(range(7)), 3))

try:
    batches([1], 0)
except ValueError as error:
    print("size 0:", error)


---
## Drill 3: Retry loop (three bugs in five lines)

**Buggy code.**

In [7]:
def call_with_retry_buggy(fn, retries=3):
    for i in range(retries):
        try:
            return fn()
        except Exception:
            pass

calls = []
def bad_request():
    calls.append(1)
    raise ValueError("400 Bad Request")

print("returned:", call_with_retry_buggy(bad_request))   # silently None
print("attempts on a 400 error:", len(calls))           # retried a request that can never succeed

returned: None
attempts on a 400 error: 3


- **Bug (a):** after the last failure it returns `None`, so the caller crashes somewhere far away with a confusing error.
  - **Fix:** re-raise the error on the last attempt.
- **Bug (b):** it retries errors that will never succeed (bad input, auth failures).
  - **Fix:** retry only on transient errors: `retry_on=(TimeoutError, ConnectionError)`.
- **Bug (c):** no wait between attempts, so 1,000 clients hammer a struggling service at the same moment (a retry storm).
  - **Fix:** exponential backoff with jitter: `base * 2**attempt + random`.
- **Tests:** always-failing → raises the original error after N attempts; `ValueError` → called once; injected `sleep` → delays grow.

In [8]:
def call_with_retry(fn, retries=3, retry_on=(TimeoutError, ConnectionError),
                    base=0.1, sleep=time.sleep):
    for attempt in range(retries):
        try:
            return fn()
        except retry_on:
            if attempt == retries - 1:
                raise
            sleep(base * 2 ** attempt + random.uniform(0, base))

calls = []
def always_timeout():
    calls.append(1)
    raise TimeoutError("boom")

no_sleep = lambda s: None
check(3, "re-raises after 3 attempts",
      raises(TimeoutError, call_with_retry, always_timeout, sleep=no_sleep) and len(calls) == 3)

calls = []
check(3, "non-retryable error tried once",
      raises(ValueError, call_with_retry, bad_request, sleep=no_sleep) and len(calls) == 1)

delays = []
raises(TimeoutError, call_with_retry, always_timeout, retries=4, base=1, sleep=delays.append)
check(3, "backoff delays grow", len(delays) == 3 and delays == sorted(delays))

✅ [3] re-raises after 3 attempts
✅ [3] non-retryable error tried once
✅ [3] backoff delays grow


In [12]:
def call_with_retry(fn, retries=3, retry_on=(TimeoutError, ConnectionError),
                    base=0.1, sleep=time.sleep):
    for attempt in range(retries):
        try:
            return fn()
        except retry_on:
            if attempt == retries - 1:
                raise
            sleep(base * 2 ** attempt + random.uniform(0, base))

def always_timeout():
    calls.append("timeout")
    raise TimeoutError("boom")

def bad_request():
    calls.append("bad request")
    raise ValueError("400 Bad Request")

# TimeoutError is in retry_on, so this tries 3 times, then raises.
calls = []
try:
    call_with_retry(always_timeout, sleep=lambda seconds: None)
except TimeoutError as error:
    print("raised:", type(error).__name__, error)
print("attempts:", len(calls), calls)

# ValueError is not in retry_on, so this tries once and raises immediately.
calls = []
try:
    call_with_retry(bad_request, sleep=lambda seconds: None)
except ValueError as error:
    print("raised:", type(error).__name__, error)
print("attempts:", len(calls), calls)

# retries=4 means 4 tries and 3 waits. Each wait is longer than the one before.
delays = []
calls = []
try:
    call_with_retry(always_timeout, retries=4, base=1, sleep=delays.append)
except TimeoutError:
    pass
print("attempts:", len(calls))
print("delays:", [round(delay, 2) for delay in delays])


raised: TimeoutError boom
attempts: 3 ['timeout', 'timeout', 'timeout']
raised: ValueError 400 Bad Request
attempts: 1 ['bad request']
attempts: 4
delays: [1.07, 2.15, 4.99]


---
## Drill 4: Race condition (check-then-act)

**Buggy code.** The `time.sleep(0.001)` only makes the race show up reliably in a demo. In production, the thread scheduler causes the same interleaving at random.

In [10]:
class RateLimiterBuggy:
    def __init__(self, limit):
        self.limit, self.counts = limit, {}

    def allow(self, user):
        n = self.counts.get(user, 0)
        if n >= self.limit:
            return False
        time.sleep(0.001)        # widens the race window for the demo
        self.counts[user] = n + 1
        return True

def hammer(limiter, threads=50):
    allowed = []
    ts = [threading.Thread(target=lambda: allowed.append(limiter.allow("u"))) for _ in range(threads)]
    [t.start() for t in ts]
    [t.join() for t in ts]
    return sum(allowed)

print("allowed with limit 10:", hammer(RateLimiterBuggy(10)))   # usually > 10

allowed with limit 10: 50


- **Bug:** reading the count and writing it back are two separate steps. Two threads can both read `n = 9`, both pass the check, and both write `10`.
- **Breaking input:** 50 concurrent requests with limit 10 → more than 10 allowed.
- **Fix:** wrap the read-modify-write in a `threading.Lock`. Across several servers, use an atomic store operation such as Redis `INCR` + `EXPIRE`.
- **Tests:** many threads → exactly `limit` allowed. Also mention that there's **no time window reset**, so it's a lifetime cap, not a rate.

In [11]:
class RateLimiter:
    def __init__(self, limit):
        self.limit, self.counts, self._lock = limit, {}, threading.Lock()

    def allow(self, user):
        with self._lock:
            n = self.counts.get(user, 0)
            if n >= self.limit:
                return False
            time.sleep(0.001)    # same race window, now harmless
            self.counts[user] = n + 1
            return True

check(4, "exactly 10 allowed under 50 threads", hammer(RateLimiter(10)) == 10)

✅ [4] exactly 10 allowed under 50 threads


---
## Drill 5: Resource leak

**Buggy code.** A tracking `open` shows whether the file handle gets closed.

In [30]:
opened = []
def tracking_open(path):
    f = open(path)
    opened.append(f)
    return f

def read_config_buggy(path, _open=tracking_open):
    f = _open(path)
    data = json.load(f)
    f.close()
    return data

with tempfile.NamedTemporaryFile("w", suffix=".json", delete=False) as tmp:
    tmp.write("{bad json")
bad_path = tmp.name

raises(json.JSONDecodeError, read_config_buggy, bad_path)
print("file left open after the error:", not opened[-1].closed)
opened[-1].close()

file left open after the error: True


- **Bug:** if `json.load` raises, `f.close()` never runs. In a long-running service, leaked handles pile up until you hit "too many open files" (the same pattern leaks DB connections from a pool).
- **Breaking input:** a malformed JSON file read in a loop.
- **Fix:** a context manager (`with open(path) as f:`) closes the file even when an exception happens.
- **Tests:** a malformed file still raises `JSONDecodeError`, **and** the handle is closed.

In [31]:
def read_config(path, _open=tracking_open):
    with _open(path) as f:
        return json.load(f)

check(5, "still raises on bad JSON", raises(json.JSONDecodeError, read_config, bad_path))
check(5, "handle closed after error", opened[-1].closed)
os.remove(bad_path)

✅ [5] still raises on bad JSON
✅ [5] handle closed after error


---
## Drill 6: N+1 queries

**Buggy code.** An in-memory SQLite database with a query counter shows the round trips.

In [29]:
class CountingDB:
    def __init__(self):
        self.conn = sqlite3.connect(":memory:")
        self.conn.execute("CREATE TABLE orders(id INTEGER PRIMARY KEY, total INTEGER)")
        self.conn.executemany("INSERT INTO orders VALUES (?, ?)", [(i, i * 10) for i in range(1, 1201)])
        self.queries = 0

    def query(self, sql, params=()):
        self.queries += 1
        return self.conn.execute(sql, params).fetchall()

def order_summaries_buggy(db, order_ids):
    return [db.query("SELECT * FROM orders WHERE id = ?", (oid,))[0] for oid in order_ids]

db = CountingDB()
order_summaries_buggy(db, list(range(1, 1201)))
print("queries for 1,200 orders:", db.queries)   # one round trip per id

queries for 1,200 orders: 1200


- **Bug:** one database round trip per id. Latency grows linearly: 5,000 ids × 2 ms = 10 seconds.
- **Breaking input:** any long list of ids, or a slow network to the DB.
- **Fix:** fetch in batches with `WHERE id IN (...)` (chunks of about 500 stay under parameter limits), or use a join. Then **re-order the results to match the input**, because SQL doesn't guarantee order.
- **Tests:** query count ≤ `ceil(n / 500)`; the result order matches the input order; missing ids are handled.

In [ ]:
def order_summaries(db, order_ids, chunk=500):
    by_id = {}
    for i in range(0, len(order_ids), chunk):
        part = order_ids[i:i + chunk]
        placeholders = ",".join("?" * len(part))
        for row in db.query(f"SELECT * FROM orders WHERE id IN ({placeholders})", part):
            by_id[row[0]] = row
    return [by_id[oid] for oid in order_ids if oid in by_id]

db = CountingDB()
ids = list(range(1200, 0, -1))           # reversed, to test ordering
result = order_summaries(db, ids)
check(6, "3 queries instead of 1,200", db.queries == 3)
check(6, "result order matches input", [r[0] for r in result] == ids)
check(6, "missing id skipped", order_summaries(CountingDB(), [1, 99999]) == [(1, 10)])

> Note: the `f"... IN ({placeholders})"` is safe because it only inserts `?` markers. The values still go in as parameters. Contrast this with Drill 11.

---
## Drill 7: Floats for money

**Buggy code.**

In [27]:
def total_buggy(prices, tax_rate):
    return round(sum(prices) * (1 + tax_rate), 2)

print(0.1 + 0.2)                       # not 0.3
print(total_buggy([0.125], 0))         # expected 0.13 (round half up), got:

0.30000000000000004
0.12


- **Bug:** binary floats can't represent most cents exactly, and Python's `round` uses *banker's rounding* (to the nearest even digit), so totals drift and half-cents round the "wrong" way for finance.
- **Breaking input:** `0.1 + 0.2` → `0.30000000000000004`; `round(0.125, 2)` → `0.12`.
- **Fix:** use `Decimal` built **from strings** and `quantize(Decimal("0.01"), ROUND_HALF_UP)`, or store integer cents.
- **Tests:** known tricky values (`0.1, 0.2`), half-cent rounding, an empty list gives `0.00`.

In [28]:
def total(prices, tax_rate):
    subtotal = sum((Decimal(p) for p in prices), Decimal("0"))
    return (subtotal * (1 + Decimal(tax_rate))).quantize(Decimal("0.01"), ROUND_HALF_UP)

check(7, "0.1 + 0.2 == 0.30", total(["0.1", "0.2"], "0") == Decimal("0.30"))
check(7, "half-cent rounds up", total(["0.125"], "0") == Decimal("0.13"))
check(7, "empty list == 0.00", total([], "0.1") == Decimal("0.00"))

✅ [7] 0.1 + 0.2 == 0.30
✅ [7] half-cent rounds up
✅ [7] empty list == 0.00


---
## Drill 8: Changing a list while looping over it

**Buggy code.**

In [25]:
def remove_inactive_buggy(users):
    for u in users:
        if not u["active"]:
            users.remove(u)
    return users

users = [{"id": 1, "active": False}, {"id": 2, "active": False}, {"id": 3, "active": True}]
print([u["id"] for u in remove_inactive_buggy(users)])   # expected [3]

[2, 3]


- **Bug:** removing an item shifts everything after it one position left, so the loop skips the next item.
- **Breaking input:** two inactive users side by side → the second one survives.
- **Fix:** build a new list: `[u for u in users if u["active"]]`. This also avoids mutating the caller's list.
- **Tests:** adjacent inactive users, all inactive, empty list, input list unchanged.

In [26]:
def remove_inactive(users):
    return [u for u in users if u["active"]]

users = [{"id": 1, "active": False}, {"id": 2, "active": False}, {"id": 3, "active": True}]
check(8, "adjacent inactive both removed", [u["id"] for u in remove_inactive(users)] == [3])
check(8, "all inactive → empty", remove_inactive([{"active": False}] * 3) == [])
check(8, "empty list", remove_inactive([]) == [])
check(8, "input list unchanged", len(users) == 3)

✅ [8] adjacent inactive both removed
✅ [8] all inactive → empty
✅ [8] empty list
✅ [8] input list unchanged


---
## Drill 9: Timezone-naive datetime (token expiry)

**Buggy code.** The demo simulates a server running in India (UTC+5:30), so it behaves the same on any machine.

In [ ]:
IST = timezone(timedelta(hours=5, minutes=30))

def is_expired_buggy(expires_at_utc_naive, local_now):
    # In production this is datetime.now(): the server's LOCAL time, also naive
    return local_now > expires_at_utc_naive

utc_now = datetime(2026, 1, 1, 12, 0, tzinfo=timezone.utc)
expires = (utc_now + timedelta(hours=2)).replace(tzinfo=None)   # valid for 2 more hours (stored as naive UTC)
local_now = utc_now.astimezone(IST).replace(tzinfo=None)        # what datetime.now() returns on an IST server

print("token valid for 2h, but expired?", is_expired_buggy(expires, local_now))   # True: wrong

- **Bug:** `datetime.now()` returns the server's **local** time without a timezone, while the stored expiry is UTC. Comparing them mixes two clocks.
- **Breaking input:** a server in IST is 5.5 hours ahead, so tokens expire 5.5 hours early. West of UTC they live too long, which is a security bug.
- **Fix:** timezone-aware datetimes everywhere: `datetime.now(timezone.utc)`, store aware timestamps, and reject naive input.
- **Tests:** inject the clock (pass `now` in); check the exact boundary at `expires_at`; a naive input raises.

In [ ]:
def is_expired(expires_at, now=None):
    if expires_at.tzinfo is None:
        raise ValueError("expires_at must be timezone-aware")
    now = now or datetime.now(timezone.utc)
    return now >= expires_at

t0 = datetime(2026, 1, 1, tzinfo=timezone.utc)
check(9, "IST server: 2h token not expired",
      is_expired(t0 + timedelta(hours=2), now=t0.astimezone(IST)) is False)
check(9, "expired exactly at the boundary", is_expired(t0, now=t0) is True)
check(9, "1s before boundary not expired", is_expired(t0, now=t0 - timedelta(seconds=1)) is False)
check(9, "naive input rejected", raises(ValueError, is_expired, datetime(2026, 1, 1)))

---
## Drill 10: Cross-tenant cache leak ⭐ (strong FDE signal)

**Buggy code.** `current_request()` stands in for a web framework's request context.

In [18]:
_current = {"tenant_id": None}
def current_request():
    return type("Req", (), dict(_current))

def fetch_report(tenant, report_id):
    return f"{tenant}'s confidential report #{report_id}"

@functools.lru_cache(maxsize=1024)
def get_report_buggy(report_id):
    tenant = current_request().tenant_id
    return fetch_report(tenant, report_id)

_current["tenant_id"] = "A"; print("Tenant A sees:", get_report_buggy(1))
_current["tenant_id"] = "B"; print("Tenant B sees:", get_report_buggy(1))   # A's data!

Tenant A sees: A's confidential report #1
Tenant B sees: A's confidential report #1


In [24]:
get_report_buggy(1)

"A's confidential report #1"

- **Bug:** the cache key is only `report_id`. The tenant is read from hidden context *inside* the cached function, so it isn't part of the key. Tenant A's result is served to tenant B.
- **Breaking input:** tenants A and B both request `report_id=1`.
- **Fix:** make the tenant part of the key, e.g. `get_report(tenant_id, report_id)`. Never read hidden request context inside a cached function.
- **Tests:** two tenants with the same id get different results. Make it a **security regression test in CI**.
- **Why it matters for Blitzy:** this reads as correct in review. Tests have to catch it, not reviewers, and that's especially true for AI-generated code.

In [21]:
@functools.lru_cache(maxsize=1024)
def get_report(tenant_id, report_id):
    return fetch_report(tenant_id, report_id)

check(10, "tenant A gets A's report", get_report("A", 1).startswith("A's"))
check(10, "tenant B gets B's report", get_report("B", 1).startswith("B's"))

✅ [10] tenant A gets A's report
✅ [10] tenant B gets B's report


In [23]:
get_report("A", 1)

"A's confidential report #1"

---
## Drill 11: SQL injection

**Buggy code.**

In [16]:
conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE users(email TEXT)")
conn.execute("INSERT INTO users VALUES ('a@x.com'), ('b@x.com')")

def find_user_buggy(conn, email):
    return conn.execute(f"SELECT * FROM users WHERE email = '{email}'").fetchall()

print(find_user_buggy(conn, "x' OR '1'='1"))   # returns EVERY user

[('a@x.com',), ('b@x.com',)]


- **Bug:** user input is pasted into the SQL text, so the input can change the query's meaning.
- **Breaking input:** `"x' OR '1'='1"` turns the filter into "always true" and returns every user.
- **Fix:** a parameterized query, `execute("... WHERE email = ?", (email,))`. The database treats the value as data, never as SQL.
- **Tests:** the injection string returns `[]`; a normal email still works.

In [17]:
def find_user(conn, email):
    return conn.execute("SELECT * FROM users WHERE email = ?", (email,)).fetchall()

check(11, "injection returns nothing", find_user(conn, "x' OR '1'='1") == [])
check(11, "normal lookup works", find_user(conn, "a@x.com") == [("a@x.com",)])

✅ [11] injection returns nothing
✅ [11] normal lookup works


---
## Drill 12: Non-idempotent webhook ⭐

**Buggy code.**

In [13]:
charges = []
def charge_customer(customer_id, amount):
    charges.append((customer_id, amount))

def handle_payment_webhook_buggy(event):
    charge_customer(event["customer_id"], event["amount"])
    return 200

event = {"id": "evt_1", "customer_id": "c1", "amount": 100}
handle_payment_webhook_buggy(event)
handle_payment_webhook_buggy(event)   # provider retried after a timeout
print("times charged:", len(charges))  # customer charged twice

times charged: 2


- **Bug:** payment providers deliver webhooks **at least once**. If your response is slow or lost, they send the same event again, and the customer is charged twice.
- **Breaking input:** the same event delivered twice.
- **Fix:** an **idempotency key**. Record `event["id"]` in a `processed` table with a unique constraint, **in the same database transaction** as the side effect. If the insert fails as a duplicate, return 200 without acting.
- **Tests:** the same event twice → charged once; two different events → both charged.
- **Interview nuance:** if the side effect is an external API call, it can't share your DB transaction. Pass the idempotency key to the provider too (Stripe supports this), and reconcile any unclear outcomes.

In [14]:
db_conn = sqlite3.connect(":memory:")
db_conn.execute("CREATE TABLE processed(event_id TEXT PRIMARY KEY)")
db_conn.execute("CREATE TABLE ledger(customer_id TEXT, amount INTEGER)")

def handle_payment_webhook(conn, event):
    try:
        with conn:   # one transaction: dedupe record + side effect commit together
            conn.execute("INSERT INTO processed(event_id) VALUES (?)", (event["id"],))
            conn.execute("INSERT INTO ledger VALUES (?, ?)", (event["customer_id"], event["amount"]))
    except sqlite3.IntegrityError:
        pass         # duplicate delivery: already processed
    return 200

handle_payment_webhook(db_conn, event)
handle_payment_webhook(db_conn, event)
check(12, "duplicate delivery charged once",
      db_conn.execute("SELECT COUNT(*) FROM ledger").fetchone()[0] == 1)
handle_payment_webhook(db_conn, {**event, "id": "evt_2"})
check(12, "different event charged",
      db_conn.execute("SELECT COUNT(*) FROM ledger").fetchone()[0] == 2)

✅ [12] duplicate delivery charged once
✅ [12] different event charged


In [ ]:
connection = sqlite3.connect(":memory:")
connection.execute("CREATE TABLE processed(event_id TEXT PRIMARY KEY)")
connection.execute("CREATE TABLE ledger(customer_id TEXT, amount INTEGER)")

def handle_payment_webhook(conn, event):
    try:
        with conn:
            conn.execute(
                "INSERT INTO processed(event_id) VALUES (?)",
                (event["id"],),
            )
            conn.execute(
                "INSERT INTO ledger VALUES (?, ?)",
                (event["customer_id"], event["amount"]),
            )
    except sqlite3.IntegrityError:
        print("already processed:", event["id"])
    return 200

def show_ledger(conn):
    rows = conn.execute("SELECT customer_id, amount FROM ledger").fetchall()
    print("ledger:", rows)

first_delivery = {"id": "evt_1", "customer_id": "c1", "amount": 100}
same_event_again = {"id": "evt_1", "customer_id": "c1", "amount": 100}
different_event = {"id": "evt_2", "customer_id": "c1", "amount": 50}

print("first delivery:", handle_payment_webhook(connection, first_delivery))
show_ledger(connection)

print("same event delivered again:", handle_payment_webhook(connection, same_event_again))
show_ledger(connection)

print("different event:", handle_payment_webhook(connection, different_event))
show_ledger(connection)


---
## Summary: all drills

In [ ]:
for drill in sorted(RESULTS):
    ok = all(RESULTS[drill])
    print(("✅" if ok else "❌"), f"Drill {drill:>2}: {sum(RESULTS[drill])}/{len(RESULTS[drill])} checks passed")
all_ok = all(all(v) for v in RESULTS.values()) and len(RESULTS) == 12
print("\nALL 12 DRILLS VERIFIED" if all_ok else "\nSOME CHECKS FAILED")

## How Blitzy could frame these

- **"How would you catch these in AI-generated code at scale?"**
  - Static analysis (Semgrep / CodeQL) for SQL injection and leaked secrets.
  - Required tests that execute real code paths, not mocks.
  - A concurrency and property-based test harness.
  - Platform **Rules**: Blitzy has rules such as *require-test-coverage* and *catalog-security-vulnerabilities*.
- **"Would a human reviewer catch Drill 10?"**
  - Often not. It reads correctly. That's why tenant-isolation tests belong in CI and aren't left to review.
- **"Fix or rewrite?"**
  - Fix minimally, add a regression test, and file the structural improvement separately so the review diff stays small.